# Notebook 05 — GGUF and llama.cpp Basics

This notebook introduces the practical local runtime layer for open models. The goal is to understand what GGUF packages, why `llama.cpp` remains important, and how to reason about local inference without hiding the moving parts.

## What you will build

- inspect the core metadata that travels with a GGUF model
- estimate memory needs across common quantizations
- map local hardware limits to practical model choices
- assemble `llama.cpp` command lines for CLI and server use
- simulate local latency so the trade-offs stay concrete

## Practical stance

We keep this notebook notebook-first and open-source only. When a full model download or native build would be awkward, we use small transparent simulations or command previews instead of pretending every environment is ready for a heavy local compile.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
random.seed(5)

ARTIFACT_DIR = ARTIFACT_ROOT / "05_gguf_and_llama_cpp_basics"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def estimate_weight_gib(params_billion, bits_per_weight):
    total_bits = params_billion * 1_000_000_000 * bits_per_weight
    return total_bits / 8 / (1024 ** 3)

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Inspect the shape of GGUF metadata

GGUF packages more than raw weights. It stores architecture details, tokenizer references, context limits, and quantization metadata in one portable artifact. That portability is why it works so well for local distribution.

In [ ]:
gguf_header = {
    "general.name": "Llama-3.1-8B-Instruct-Q4_K_M",
    "general.architecture": "llama",
    "general.file_type": "Q4_K_M",
    "llama.context_length": 8192,
    "llama.embedding_length": 4096,
    "llama.block_count": 32,
    "tokenizer.ggml.model": "gpt2",
    "tokenizer.ggml.add_bos_token": True,
    "quantize.imatrix.entries_count": 224,
    "source.repo": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
}

metadata_rows = [{"key": key, "value": value} for key, value in gguf_header.items()]
gguf_metadata_df = pd.DataFrame(metadata_rows)
gguf_metadata_df

## Step 2: Estimate weight memory across quantizations

The first fast question for a local runtime is simple: do the weights fit? The table below uses approximate bits-per-weight values. That is enough for planning, even before you benchmark a specific build.

In [ ]:
quant_profiles = [
    {"quant": "Q4_K_M", "bits_per_weight": 4.83, "quality_note": "strong default local trade-off", "cpu_decode_tps": 22, "gpu_bonus_tps": 20},
    {"quant": "Q5_K_M", "bits_per_weight": 5.52, "quality_note": "slightly better quality, modestly larger", "cpu_decode_tps": 19, "gpu_bonus_tps": 18},
    {"quant": "Q8_0", "bits_per_weight": 8.50, "quality_note": "heavy but usually closer to fp16", "cpu_decode_tps": 13, "gpu_bonus_tps": 12},
    {"quant": "F16", "bits_per_weight": 16.0, "quality_note": "reference quality, rarely the best laptop choice", "cpu_decode_tps": 7, "gpu_bonus_tps": 10},
]

model_sizes_b = [3, 7, 8, 13]
weight_rows = []
for size_b in model_sizes_b:
    for profile in quant_profiles:
        weight_rows.append({
            "model_b": size_b,
            "quant": profile["quant"],
            "weight_gib_est": round(estimate_weight_gib(size_b, profile["bits_per_weight"]), 2),
            "quality_note": profile["quality_note"],
        })

weight_df = pd.DataFrame(weight_rows)
weight_df

## Step 3: Map model choices to local hardware

Local inference is mostly a budgeting exercise. Below we ask a practical question: which 8B variants fit on common local machines if we reserve headroom for the OS, caches, and notebook overhead?

In [ ]:
hardware_profiles = [
    {"machine": "thin_laptop_cpu", "ram_gib": 16, "free_vram_gib": 0},
    {"machine": "desktop_cpu", "ram_gib": 32, "free_vram_gib": 0},
    {"machine": "consumer_gpu_12g", "ram_gib": 32, "free_vram_gib": 12},
    {"machine": "consumer_gpu_24g", "ram_gib": 64, "free_vram_gib": 24},
]

fit_rows = []
target_rows = weight_df[weight_df["model_b"] == 8].to_dict("records")
for machine in hardware_profiles:
    for row in target_rows:
        free_system_ram = machine["ram_gib"] * 0.72
        fits_cpu = row["weight_gib_est"] < free_system_ram
        fits_gpu = row["weight_gib_est"] < max(machine["free_vram_gib"] - 1.5, 0)
        fit_rows.append({
            "machine": machine["machine"],
            "quant": row["quant"],
            "weight_gib_est": row["weight_gib_est"],
            "fits_cpu_only": fits_cpu,
            "fits_full_gpu": fits_gpu,
        })

fit_df = pd.DataFrame(fit_rows)
fit_df

## Step 4: Plan GGUF acquisition

In practice you usually download a specific GGUF file from Hugging Face and keep it in a local model cache. The notebook builds the exact command strings so you can review them before running anything large.

In [ ]:
download_targets = [
    {
        "repo": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        "filename": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    },
    {
        "repo": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "filename": "Qwen2.5-7B-Instruct-Q5_K_M.gguf",
    },
]

download_rows = []
for target in download_targets:
    local_dir = CACHE_DIR / "gguf"
    cli_command = [
        "huggingface-cli", "download", target["repo"], target["filename"],
        "--local-dir", str(local_dir),
    ]
    download_rows.append({
        "repo": target["repo"],
        "filename": target["filename"],
        "command_preview": " ".join(cli_command),
    })

download_df = pd.DataFrame(download_rows)
download_df

## Step 5: Understand the main `llama.cpp` surfaces

Three entry points matter most at the start:

- `llama-cli` for direct prompting and inspection
- `llama-server` for local API serving
- language bindings such as `llama-cpp-python` when you want everything inside Python

In [ ]:
def llama_cli_command(model_path, prompt, ctx_size=4096, threads=8, n_gpu_layers=0, temp=0.2):
    return [
        "llama-cli",
        "-m", str(model_path),
        "-c", str(ctx_size),
        "-t", str(threads),
        "-ngl", str(n_gpu_layers),
        "--temp", str(temp),
        "-p", prompt,
    ]

sample_cli = llama_cli_command(
    model_path=CACHE_DIR / "gguf" / "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    prompt="Summarize why GGUF is convenient for local inference.",
    ctx_size=8192,
    threads=10,
    n_gpu_layers=20,
)

print("CLI preview:")
print(" ".join(sample_cli))

## Step 6: Format a chat transcript for a local prompt

Many local workflows still require explicit prompt construction. Seeing the prompt helps you debug stop tokens, repeated system messages, and template mismatches.

In [ ]:
messages = [
    {"role": "system", "content": "You are a practical systems tutor."},
    {"role": "user", "content": "Explain when Q4_K_M is a good local choice."},
]

def render_chat_prompt(messages):
    rendered = []
    for message in messages:
        rendered.append(f'[{message["role"].upper()}]')
        rendered.append(message["content"])
    rendered.append("[ASSISTANT]")
    return "\n".join(rendered)

chat_prompt = render_chat_prompt(messages)
print(chat_prompt)

## Step 7: Reason about CPU and GPU offload

A common `llama.cpp` decision is how many layers to offload. Full offload is great when it fits. Partial offload can still help, but only if you leave enough VRAM headroom.

In [ ]:
def recommend_n_gpu_layers(total_layers, model_weight_gib, free_vram_gib, reserve_gib=1.5):
    usable_vram = max(free_vram_gib - reserve_gib, 0)
    ratio = min(1.0, usable_vram / max(model_weight_gib, 0.01))
    return int(round(total_layers * ratio))

offload_rows = []
q4_weight_gib = float(weight_df[(weight_df["model_b"] == 8) & (weight_df["quant"] == "Q4_K_M")]["weight_gib_est"].iloc[0])
for machine in hardware_profiles:
    offload_rows.append({
        "machine": machine["machine"],
        "recommended_n_gpu_layers": recommend_n_gpu_layers(32, q4_weight_gib, machine["free_vram_gib"]),
        "free_vram_gib": machine["free_vram_gib"],
    })

pd.DataFrame(offload_rows)

## Step 8: Simulate local latency

We do not need a full model load to teach the main idea. Prefill cost grows with prompt length. Decode cost grows with generated tokens. Faster quantizations and more GPU offload mostly improve token throughput.

In [ ]:
def simulate_local_run(prompt_tokens, output_tokens, profile, n_gpu_layers):
    prefill_tps = 180 + n_gpu_layers * 16
    decode_tps = profile["cpu_decode_tps"] + min(n_gpu_layers, 24) * (profile["gpu_bonus_tps"] / 24)
    prefill_s = prompt_tokens / prefill_tps
    decode_s = output_tokens / max(decode_tps, 1)
    total_s = prefill_s + decode_s
    return {
        "prefill_s": round(prefill_s, 3),
        "decode_s": round(decode_s, 3),
        "total_s": round(total_s, 3),
        "tokens_per_second": round(output_tokens / max(total_s, 0.001), 2),
    }

simulation_rows = []
for prompt_tokens in [128, 512, 1024]:
    for profile in quant_profiles[:3]:
        metrics = simulate_local_run(prompt_tokens, 160, profile, n_gpu_layers=20)
        simulation_rows.append({
            "prompt_tokens": prompt_tokens,
            "quant": profile["quant"],
            **metrics,
        })

simulation_df = pd.DataFrame(simulation_rows)
simulation_df

## Step 9: Visualize the latency pattern

The chart makes the local lesson visible: longer prompts make prefill dominant, while slower quantizations mainly stretch decode time.

In [ ]:
pivot = simulation_df.pivot(index="prompt_tokens", columns="quant", values="total_s")
ax = pivot.plot(kind="bar", figsize=(8, 4), rot=0, title="Estimated total latency by prompt length and quantization")
ax.set_ylabel("seconds")
plt.tight_layout()

## Step 10: Prepare a local OpenAI-compatible serving command

Even if you start from the CLI, the next operational step is often a local HTTP server. `llama-server` provides an easy bridge into application code that already expects OpenAI-style request shapes.

In [ ]:
def llama_server_command(model_path, host="127.0.0.1", port=8000, ctx_size=8192, n_gpu_layers=20):
    return [
        "llama-server",
        "-m", str(model_path),
        "--host", host,
        "--port", str(port),
        "-c", str(ctx_size),
        "-ngl", str(n_gpu_layers),
    ]

server_command = llama_server_command(CACHE_DIR / "gguf" / "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf")
sample_request = {
    "model": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    "messages": [
        {"role": "system", "content": "You are a concise systems tutor."},
        {"role": "user", "content": "Explain the purpose of GGUF in two sentences."},
    ],
    "temperature": 0.2,
    "max_tokens": 128,
}

print("Server command preview:")
print(" ".join(server_command))
print("\nRequest payload preview:")
print(json.dumps(sample_request, indent=2))

## Step 11: Troubleshooting checklist

When local inference feels slow or unstable, inspect the basics first:

- confirm the model path and quantization filename
- check whether the context window is larger than you intended
- inspect thread count and GPU offload settings
- measure prompt length separately from output length
- compare CPU-only and partial-offload runs before changing many variables at once

In [ ]:
manifest = {
    "notebook": "05_gguf_and_llama_cpp_basics",
    "recommended_defaults": {
        "starter_quant": "Q4_K_M",
        "ctx_size": 8192,
        "notes": "Use simulations first, then replace with measured runs on your machine.",
    },
    "download_plan": download_rows,
    "server_command": server_command,
}

manifest_path = ARTIFACT_DIR / "gguf_basics_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote manifest:", manifest_path.resolve())

## Recap

You now have a clear local baseline: GGUF makes models portable, `llama.cpp` exposes the runtime knobs directly, and local planning starts with memory budgets before it becomes a throughput experiment. The next notebook looks more closely at quantization and memory trade-offs.